# 1. Libraries & Sample Data
The first step is to load our Python Libraries and download the sample data. The dataset represents Apple stock price (1d bars) for the year 2010

In [ ]:
# Load Python Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from IPython.display import display, HTML

# for dataframe display
pd.set_option("display.max_rows", None)


def display_df(df):
    # Puts the scrollbar next to the DataFrame
    display(
        HTML(
            "<div style='height: 200px; overflow: auto; width: fit-content'>"
            + df.to_html()
            + "</div>"
        )
    )

In [ ]:
# Download Sample Data AAPL_2009-2010_6m_features_1d.csv
apple_filename = "./../e2_data_CLEAN.csv"
data = pd.read_csv(apple_filename, index_col=0)
# Use pandas's to_datetime() to convert any columns that are in a datetime format
data["Date"] = pd.to_datetime(data["Date"])
data.set_index("Date", inplace=True)

In [ ]:
data.info()

In [ ]:
# Plot the Close Data
data["Close"].plot()

In [ ]:
# Check for null values
print("Number of Null Values =\n", data.isnull().sum())

In [ ]:
data.head()

# 5. State Space Representation
Now we have a set of data with OHLC data plus some techinchal indicators. Using this data, construct the state space matrix, whese features of the state space are Close Price, 5-day Moving Average, 20-day Moving Average, Bollinger Bands (upper and lower), and 20-day Historical Volatility of Close Price.

In [ ]:
# Construct the State Space Matrix
dataset = data[["Adj Close"]].copy()
dataset = dataset.rename(columns={"Adj Close": "Close"})
dataset["MA5"] = dataset["Close"].rolling(window=5).mean()
dataset["MA20"] = dataset["Close"].rolling(window=20).mean()
dataset["Vol20"] = dataset["Close"].rolling(window=20).std()
dataset["BB_upper"] = dataset["MA20"] + 2 * dataset["Vol20"]
dataset["BB_lower"] = dataset["MA20"] - 2 * dataset["Vol20"]
display_df(dataset)

In [ ]:
dataset.head(40)

# 6. Z-Score Normalization
Now that we have cleaned our data, and created our features of interest, we must normalize our data. For this example, we use the sklearn StandardScaler, which centers the data and normalizes to unit variance (i.e. performs z-score normalization for us). Do this in a simple, non-rolling fashion.

In [ ]:
# Plot Un-normalized Close Price and Bollinger Bands
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    dataset.index, dataset["Close"], label="Close Price", color="blue", linewidth=1.5
)
ax.plot(
    dataset.index,
    dataset["MA20"],
    label="MA20",
    color="orange",
    linestyle="--",
    linewidth=1.2,
)
ax.plot(
    dataset.index,
    dataset["BB_upper"],
    label="BB Upper (MA20 + 2σ)",
    color="green",
    linestyle=":",
    linewidth=1.2,
)
ax.plot(
    dataset.index,
    dataset["BB_lower"],
    label="BB Lower (MA20 - 2σ)",
    color="red",
    linestyle=":",
    linewidth=1.2,
)
ax.fill_between(
    dataset.index,
    dataset["BB_lower"],
    dataset["BB_upper"],
    alpha=0.1,
    color="gray",
    label="Bollinger Band Range",
)

ax.set_title("AAPL Close Price with Bollinger Bands (Un-normalized)", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend(loc="upper left")
ax.grid(True, linestyle="--", alpha=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Normalize Dataset with StandardScaler
normlist = []
static_normed_dataset = pd.DataFrame(index=dataset.index, columns=dataset.columns)
for col in dataset.columns:
    if col == "Date":
        static_normed_dataset[col] = dataset[col]
        continue
    normalizer = StandardScaler()
    column_data = pd.DataFrame(dataset[col])
    # fit normalizer, transform data column with fitted normalizer, put transformed column in normed dataframe
    normalizer.fit(column_data)
    static_normed_dataset[col] = normalizer.transform(column_data).flatten()
    normlist.append(normalizer)

In [ ]:
# Plot Normalized Features: Close, MA20, BB Upper, BB Lower
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    static_normed_dataset.index,
    static_normed_dataset["Close"],
    label="Close Price",
    color="blue",
    linewidth=1.5,
)
ax.plot(
    static_normed_dataset.index,
    static_normed_dataset["MA20"],
    label="MA20",
    color="orange",
    linestyle="--",
    linewidth=1.2,
)
ax.plot(
    static_normed_dataset.index,
    static_normed_dataset["BB_upper"],
    label="BB Upper (MA20 + 2σ)",
    color="green",
    linestyle=":",
    linewidth=1.2,
)
ax.plot(
    static_normed_dataset.index,
    static_normed_dataset["BB_lower"],
    label="BB Lower (MA20 - 2σ)",
    color="red",
    linestyle=":",
    linewidth=1.2,
)
ax.fill_between(
    static_normed_dataset.index,
    static_normed_dataset["BB_lower"],
    static_normed_dataset["BB_upper"],
    alpha=0.1,
    color="gray",
    label="Bollinger Band Range",
)

ax.set_title("AAPL Close Price with Bollinger Bands (Z-Score Normalized)", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Z-Score")
ax.legend(loc="upper left")
ax.grid(True, linestyle="--", alpha=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Normalized Features: Close, MA20, MA5
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    static_normed_dataset.index,
    static_normed_dataset["Close"],
    label="Close Price",
    color="blue",
    linewidth=1.5,
)
ax.plot(
    static_normed_dataset.index,
    static_normed_dataset["MA20"],
    label="MA20 (20-day)",
    color="orange",
    linestyle="--",
    linewidth=1.2,
)
ax.plot(
    static_normed_dataset.index,
    static_normed_dataset["MA5"],
    label="MA5 (5-day)",
    color="green",
    linestyle="-.",
    linewidth=1.2,
)

ax.set_title("AAPL Close Price with MA5 and MA20 (Z-Score Normalized)", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Z-Score")
ax.legend(loc="upper left")
ax.grid(True, linestyle="--", alpha=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Normalized Features: Close, Volatility
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    static_normed_dataset.index,
    static_normed_dataset["Close"],
    label="Close Price",
    color="blue",
    linewidth=1.5,
)
ax.plot(
    static_normed_dataset.index,
    static_normed_dataset["Vol20"],
    label="20-day Historical Volatility",
    color="orange",
    linestyle="--",
    linewidth=1.2,
)

ax.set_title(
    "AAPL Close Price with 20-day Historical Volatility (Z-Score Normalized)",
    fontsize=14,
)
ax.set_xlabel("Date")
ax.set_ylabel("Z-Score")
ax.legend(loc="upper left")
ax.grid(True, linestyle="--", alpha=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 7. Rolling Z-Score Normalization
Now that we have cleaned our data, and created our features of interest, we must normalize our data. For this example, we use the sklearn StandardScaler, which centers the data and normalizes to unit variance. Due to the fact that our data is time-series data, it is best practice to do this in a rolling fashion. We choose 20 days as our window for normalization, and run the StandardScaler in a rolling (non-overlapping) fashion. 

In [ ]:
# Display raw dataset (unnormalized)
display_df(dataset)

In [ ]:
# Normalize the chosen price data & features
normed_dataset = pd.DataFrame(index=dataset.index, columns=dataset.columns)
step = 20
for col in dataset.columns:
    n = 0
    if col == "Date":
        normed_dataset[col] = dataset[col]
        continue
    while n <= len(dataset.index):
        normalizer = StandardScaler()
        if n == 0:
            column_data = dataset[col].iloc[:step]
        elif n + step >= len(dataset.index):
            column_data = dataset[col].iloc[n:]
        else:
            column_data = dataset[col].iloc[n : n + step]
        normalizer.fit(column_data.values.reshape(-1, 1))
        normed_dataset.loc[column_data.index, col] = normalizer.transform(
            column_data.values.reshape(-1, 1)
        ).flatten()
        n += step
normed_dataset = normed_dataset.astype(float)
display_df(normed_dataset)

In [ ]:
# Plot Normalized Features: Close, MA20, BB Upper, BB Lower
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    normed_dataset.index,
    normed_dataset["Close"],
    label="Close Price",
    color="blue",
    linewidth=1.5,
)
ax.plot(
    normed_dataset.index,
    normed_dataset["MA20"],
    label="MA20",
    color="orange",
    linestyle="--",
    linewidth=1.2,
)
ax.plot(
    normed_dataset.index,
    normed_dataset["BB_upper"],
    label="BB Upper (MA20 + 2σ)",
    color="green",
    linestyle=":",
    linewidth=1.2,
)
ax.plot(
    normed_dataset.index,
    normed_dataset["BB_lower"],
    label="BB Lower (MA20 - 2σ)",
    color="red",
    linestyle=":",
    linewidth=1.2,
)
ax.fill_between(
    normed_dataset.index,
    normed_dataset["BB_lower"],
    normed_dataset["BB_upper"],
    alpha=0.1,
    color="gray",
    label="Bollinger Band Range",
)

ax.set_title(
    "AAPL Close Price with Bollinger Bands (Rolling Z-Score Normalized)", fontsize=14
)
ax.set_xlabel("Date")
ax.set_ylabel("Z-Score")
ax.legend(loc="upper left")
ax.grid(True, linestyle="--", alpha=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Normalized Features: Close, MA20, MA5
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    normed_dataset.index,
    normed_dataset["Close"],
    label="Close Price",
    color="blue",
    linewidth=1.5,
)
ax.plot(
    normed_dataset.index,
    normed_dataset["MA20"],
    label="MA20 (20-day)",
    color="orange",
    linestyle="--",
    linewidth=1.2,
)
ax.plot(
    normed_dataset.index,
    normed_dataset["MA5"],
    label="MA5 (5-day)",
    color="green",
    linestyle="-.",
    linewidth=1.2,
)

ax.set_title(
    "AAPL Close Price with MA5 and MA20 (Rolling Z-Score Normalized)", fontsize=14
)
ax.set_xlabel("Date")
ax.set_ylabel("Z-Score")
ax.legend(loc="upper left")
ax.grid(True, linestyle="--", alpha=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Normalized Features: Close, Volatility
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    normed_dataset.index,
    normed_dataset["Close"],
    label="Close Price",
    color="blue",
    linewidth=1.5,
)
ax.plot(
    normed_dataset.index,
    normed_dataset["Vol20"],
    label="20-day Historical Volatility",
    color="orange",
    linestyle="--",
    linewidth=1.2,
)

ax.set_title(
    "AAPL Close Price with 20-day Historical Volatility (Rolling Z-Score Normalized)",
    fontsize=14,
)
ax.set_xlabel("Date")
ax.set_ylabel("Z-Score")
ax.legend(loc="upper left")
ax.grid(True, linestyle="--", alpha=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()